# TBX11K Explanation Cases (radiologist review)

Generates natural-language explanations for a hand-picked mix of TBX11K images, for the
radiologist face-validity consultation. The goal is to check whether the explanation
*phrasing* reads the way a clinician would describe the finding.

**Honest caveats (the explanation module is not yet wired to the trained models):**

- The JSON records here are built from TBX11K **ground-truth annotations**, not from our
  classifier and detector outputs. The end-to-end integration is pending, so this notebook
  stands in for it by feeding the true labels and boxes into the explanation module.
- Class **probabilities are illustrative placeholders** encoding the ground-truth label,
  not real model scores. Region **confidence is set to high** because these are
  expert-annotated true regions.
- Laterality follows radiological convention: image-left is the patient's right.


In [ ]:
import sys, csv, ast, collections
from pathlib import Path

sys.path.insert(0, "../src")
from tb_explain import adapt, explain

DATA = Path("../../../dataset/tbx11k-simplified")
IMAGES = DATA / "images"

# Load annotations, grouped by image (one CSV row per bounding box).
rows = list(csv.DictReader(open(DATA / "data.csv")))
images = collections.defaultdict(lambda: {"boxes": [], "image_type": None, "source": None})
for r in rows:
    d = images[r["fname"]]
    d["image_type"] = r["image_type"]
    d["source"] = r["source"]
    if r["bbox"] != "none":
        b = ast.literal_eval(r["bbox"])
        d["boxes"].append((r["tb_type"], b))

# Illustrative class probabilities standing in for real classifier scores.
# They encode the ground-truth label and are NOT model outputs.
PROBS = {
    "tb":             {"healthy": 0.03, "sick_non_tb": 0.07, "tb": 0.90},
    "healthy":        {"healthy": 0.92, "sick_non_tb": 0.06, "tb": 0.02},
    "sick_but_no_tb": {"healthy": 0.10, "sick_non_tb": 0.85, "tb": 0.05},
}

def build_record(fname):
    """Build a detector-format record from TBX11K ground-truth annotations."""
    d = images[fname]
    detections = [
        ((b["xmin"], b["ymin"], b["xmin"] + b["width"], b["ymin"] + b["height"]), tb_type, 0.9)
        for tb_type, b in d["boxes"]
    ]  # GT regions -> high confidence band
    return adapt(PROBS[d["image_type"]], detections, (512, 512))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

BOX_COLOR = {"active_tb": "#e3342f", "latent_tb": "#f6993f"}

def draw_case(ax, fname):
    img = Image.open(IMAGES / fname).convert("L")
    W, H = img.size
    ax.imshow(img, cmap="gray")
    # 3x3 grid used for the location field
    for k in (1, 2):
        ax.axhline(H * k / 3, color="cyan", lw=0.5, alpha=0.4)
        ax.axvline(W * k / 3, color="cyan", lw=0.5, alpha=0.4)
    # radiological laterality (image-left is the patient's right)
    ax.text(0.02 * W, 0.05 * H, "Patient R", color="cyan", fontsize=8, va="top")
    ax.text(0.98 * W, 0.05 * H, "Patient L", color="cyan", fontsize=8, va="top", ha="right")
    for tb_type, b in images[fname]["boxes"]:
        ax.add_patch(patches.Rectangle((b["xmin"], b["ymin"]), b["width"], b["height"],
                                       fill=False, edgecolor=BOX_COLOR[tb_type], lw=2))
        ax.text(b["xmin"], b["ymin"] - 4, tb_type.replace("_", " "),
                color=BOX_COLOR[tb_type], fontsize=8, va="bottom")
    ax.set_xticks([]); ax.set_yticks([])


## Selected cases

A mix spanning single and multiple regions, active and latent TB, both lungs, and the
no-finding classes. Laterality varies so the radiologist can check the left/right convention.
Edit `CASES` to swap in other image IDs from `data.csv`.


In [ ]:
CASES = [
    ("tb0009.png", "Single active TB, patient's left upper zone"),
    ("tb0017.png", "Single latent TB, patient's right upper zone"),
    ("tb0006.png", "Bilateral active TB (both mid zones)"),
    ("tb0135.png", "Mixed: two latent and one active, three regions"),
    ("tb0004.png", "Two active regions, upper zones"),
    ("s0001.png",  "Sick but not TB, no TB regions"),
    ("h0001.png",  "Healthy, no findings"),
]


## Preview images and records (no model load, runs instantly)

In [ ]:
for fname, note in CASES:
    print("=" * 70)
    print(fname, "-", note)
    print(build_record(fname).model_dump_json(indent=2))

fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for ax, (fname, note) in zip(axes.ravel(), CASES):
    draw_case(ax, fname)
    ax.set_title(f"{fname}\n{note}", fontsize=8)
for ax in axes.ravel()[len(CASES):]:
    ax.axis("off")
plt.tight_layout(); plt.show()


## Generate explanations

The first `explain()` call downloads and loads Phi-3.5-mini-instruct (several GB) and is slow.
Later calls reuse the cached model.

In [ ]:
summaries = {}
for fname, note in CASES:
    summaries[fname] = explain(build_record(fname))
    print("=" * 70)
    print(fname, "-", note)
    print(summaries[fname])
    print()


## Printable case sheets

One sheet per case (radiograph with boxes on the left, record and generated explanation on
the right), saved to `../outputs/radiologist_cases/` for printing.

In [ ]:
import textwrap
OUT = Path("../outputs/radiologist_cases"); OUT.mkdir(parents=True, exist_ok=True)

for fname, note in CASES:
    record = build_record(fname)
    summary = summaries.get(fname, "(run the generation cell first)")
    fig, (ax_img, ax_txt) = plt.subplots(1, 2, figsize=(12, 6))
    draw_case(ax_img, fname)
    ax_img.set_title(f"{fname}  -  {note}", fontsize=9)
    ax_txt.axis("off")
    lines = [f"Case: {fname}", note, "",
             f"Predicted label: {record.image_classification.predicted_label}", ""]
    if record.regions:
        lines.append("Marked regions:")
        for reg in record.regions:
            lines.append(f"  - {reg.type.replace('_',' ')}, "
                         f"{reg.location.replace('_',' ')}, {reg.confidence_band} confidence")
    else:
        lines.append("Marked regions: none")
    lines += ["", "Generated explanation:", ""]
    lines += textwrap.wrap(summary, 52)
    ax_txt.text(0, 1, "\n".join(lines), va="top", ha="left", fontsize=10, family="monospace")
    fig.tight_layout()
    fig.savefig(OUT / f"{fname.replace('.png','')}_sheet.png", dpi=150, bbox_inches="tight")
    plt.show()
print("Saved sheets to", OUT.resolve())
